# 04 AI カメラ — IMX500 でオンセンサー物体検出

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 解説（この文章）→ コード → 結果（画像）→ ✅ チェックポイント、の順で進みます。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

> ### ⚠️ 先に AI カメラのモデルを入れてください
> このノートは **AI カメラ用モデル `imx500-all`** が必要です。**Pi のターミナル**（SSH）で一度だけ実行します（約1分・オンライン）:
>
> ```bash
> sudo apt install -y imx500-all
> ```
>
> - もし `Firmware file /usr/share/imx500-models/....rpk does not exist.` と出たら、**これが未導入**という意味です。上のコマンドで入れてから、このノートを上から実行し直してください。
> - この Pi は**再起動すると配布時の状態に戻る**ため、再起動後はもう一度このコマンドが必要です。

**目標**：**Raspberry Pi AI カメラ（Sony IMX500）**で、カメラの**センサ内部で**学習済み AI モデルを動かし、物体検出を行う。OpenCV（ルールベース）と AI（学習ベース）の違いを手を動かして体感する。

> **👥 カメラは 2人1組で共有**（操作役／観察役を交代）。AI カメラは**保存画像では動かせず実機が必要**なので、ペアで 1 台を回して**いろいろな物にかざして**みましょう。

> **🎯 このノートのゴール**
>
> - [ ] 「エッジ AI」「オンセンサー推論」が何で、なぜ嬉しいかを説明できる
> - [ ] IMX500 に学習済みモデルを載せ、物体検出を実行できた
> - [ ] 検出結果（ラベル・確信度・枠）を OpenCV で画像に描けた

## 4-1. なぜ「カメラの中で AI」？
普通の AI 画像認識は「カメラで撮る → 画像を PC やクラウドへ送る → そこで AI が推論」という流れ。
**Raspberry Pi AI カメラ**は、Sony **IMX500** という **AI 処理回路（NPU）をセンサ自体に内蔵**し、**撮影と推論をセンサの中で同時に**行い、Pi には**「結果（何がどこにあるか）」だけ**を返します（＝**エッジ AI**）。

| オンセンサー AI の利点 | なぜ現場で効くか |
|---|---|
| **低遅延** | 画像転送・外部推論が要らず、結果がすぐ返る |
| **低消費電力** | Pi の CPU/GPU をほぼ使わない |
| **省帯域** | 画像を送らず**結果（数値）だけ**なのでネットワークが軽い |
| **プライバシー** | **生画像を外に出さず**に「人が何人いるか」等だけ取り出せる |

> **OpenCV と AI カメラの違い**：01–03 の OpenCV は「こう処理しろ」と**人がルールを書く**画像処理。AI カメラは「これは何か」を**学習済みモデルが判断**します。どちらが上ではなく、**前処理は OpenCV、認識は AI**、と**組み合わせる**のが実務の王道です（次の 05 で統合します）。

## 4-2. モデルを載せてカメラを起動
初回は **モデルをセンサへ転送**するため数十秒かかります（進捗バーが出ます。慌てて止めない）。

> 次のセルで `Firmware file ... .rpk does not exist.` が出たら、冒頭の `sudo apt install -y imx500-all` がまだです。

In [ ]:
import cv2, time
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'  # 日本語ラベルの豆腐対策（要 fonts-noto-cjk）
from IPython.display import clear_output
%matplotlib inline
from picamera2 import Picamera2
from picamera2.devices import IMX500

MODEL = '/usr/share/imx500-models/imx500_network_ssd_mobilenetv2_fpnlite_320x320_pp.rpk'
imx = IMX500(MODEL)
labels = imx.network_intrinsics.labels   # COCO 90 ラベル
picam = Picamera2(imx.camera_num)
picam.configure(picam.create_preview_configuration(
    main={'size':(640,480),'format':'RGB888'}, controls={'FrameRate':15}, buffer_count=8))
imx.show_network_fw_progress_bar(); picam.start(); time.sleep(2)
print('AIカメラ準備OK / ラベル数:', len(labels))

## 4-3. 物体検出（ライブで枠とラベル）
人・椅子・カップ・ペットボトル・本など、**COCO の 90 種類**にカメラを向けてみましょう。

> **`convert_inference_coords` が座標合わせを肩代わり**：AI モデルは内部で 320×320 を見ていますが表示は 640×480。この座標のズレを `imx.convert_inference_coords(box, md_, picam)` が**表示画像に合った (x,y,w,h)** に直してくれます。

In [ ]:
THRESH = 0.4
N = 40
for i in range(N):
    md_ = picam.capture_metadata()
    outputs = imx.get_outputs(md_, add_batch=True)
    frame = picam.capture_array()
    if outputs is not None:
        boxes, scores, classes = outputs[0][0], outputs[1][0], outputs[2][0]
        for box, score, cls in zip(boxes, scores, classes):
            if score < THRESH: continue
            x,y,w,h = imx.convert_inference_coords(box, md_, picam)
            cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
            cv2.putText(frame, f'{labels[int(cls)]} {score:.2f}', (x,y-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
    clear_output(wait=True)
    plt.figure(figsize=(7,5)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'AI detect {i+1}/{N}'); plt.show()
print('終了')

✅ **チェックポイント**：身近な物に枠とラベル（例 `person 0.82`）が付けば成功です。

- 検出が出ない → 被写体を変える / `THRESH` を 0.3 に下げる。
- `THRESH` を 0.6 に上げると、確信度の高いものだけになります。
- モデルを差し替えれば姿勢推定・分類・領域分割にも切り替わります（06 で扱います）。

> 🤖 **ChatGPT に聞いてみよう**
> 「IMX500 のような**オンセンサーAI（エッジAI）**は、クラウドでAIを動かすのと比べて何が嬉しい？ 具体例つきで」と聞いて、今日のキモを自分の言葉にしてみよう。

## 4-9. 後始末（必ず実行）

In [ ]:
picam.close(); print('カメラ解放')

## ✅ 到達確認

- [ ] オンセンサー AI（エッジ AI）の利点を 30 秒で説明できる
- [ ] 物体検出ができ、枠とラベルを描けた
- [ ] 後始末（`picam.close()`）を実行した